# 02 - Dumpsite box detector (Aerial-Dumping-Sites, Roboflow v6)

**Question:** *where exactly* in an image are the dump sites? (boxes)

* Data: Roboflow Universe "Aerial-Dumping-Sites" v6 (CC BY 4.0), 1,492 train / 63 valid images, one class.
* **Leakage fixes:** Roboflow stores augmented copies of one photo as `<name>_jpg.rf.<hash>.jpg`; copies are kept
  together when splitting, and training images that are near-duplicates of test images are removed.
* The 63 official validation images are our **test set, scored once** - the same images as the earlier YOLOv8s
  baseline (mAP50 0.42, which was chosen on those images, so it is slightly optimistic).
* Model: torchvision Faster R-CNN ResNet-50-FPN v2 (BSD licence; Ultralytics YOLO is AGPL-3.0).

In [ ]:
# Setup: GPU, Google Drive (click "Connect to Google Drive" when asked), code
import os, sys, subprocess, json, pathlib
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
from google.colab import drive
drive.mount("/content/drive")
RUNS = pathlib.Path("/content/drive/MyDrive/waste_runs"); RUNS.mkdir(parents=True, exist_ok=True)
REPO = "/content/waste_detection"
if not os.path.exists(REPO):
    !git clone -q https://github.com/inayatarshad/waste_detection.git {REPO}
else:
    !git -C {REPO} pull -q
!pip install -q -r {REPO}/requirements.txt
sys.path.insert(0, f"{REPO}/src")
import numpy as np, torch
print("torch", torch.__version__, "| GPU:", torch.cuda.is_available())

In [ ]:
# Download from Roboflow (needs a Roboflow API key: roboflow.com -> Settings -> API key)
!pip install -q roboflow
from getpass import getpass
from roboflow import Roboflow
DATA = pathlib.Path("/content/dumpsites")
if not (DATA / "train").exists():
    rf = Roboflow(api_key=getpass("Roboflow API key: "))
    rf.workspace("object-detection-of-illegal-dumping-sites").project("aerial-dumping-sites").version(6).download(
        "yolov8", location=str(DATA))
!ls {DATA}

In [ ]:
# Splits with leakage checks
from wastedet import dumpsites as DS
from wastedet.imageio_utils import load_rgb, dhash, hamming
from tqdm.auto import tqdm
train_all = DS.load_split(DATA, "train"); test = DS.load_split(DATA, "valid")
print(f"train {len(train_all)} images from {len({r.group for r in train_all})} originals; test {len(test)}")
train_all, n_same = DS.drop_overlap(train_all, test)
h_test = [dhash(load_rgb(r.image)) for r in tqdm(test)]
near = {r.group for r in tqdm(train_all) if min(hamming(dhash(load_rgb(r.image)), h) for h in h_test) <= 4}
train_all = [r for r in train_all if r.group not in near]
print(f"removed {n_same} train images sharing an original with test, and {len(near)} near-duplicate originals")
train, val = DS.grouped_split(train_all, val_frac=0.15, seed=0)
print(f"final: train {len(train)}, val {len(val)}, test {len(test)}  |  boxes in test: {sum(len(r.boxes) for r in test)}")

In [ ]:
from wastedet.train_det import DetConfig, train_detector
cfg = DetConfig(size=800, epochs=24, batch_size=4, lr=0.01, num_workers=2)
OUT = RUNS / "detector_v1"
res = train_detector(cfg, DS.DumpsiteDetDataset(train, cfg.size, train=True), DS.DumpsiteDetDataset(val, cfg.size), OUT)

In [ ]:
# Test ONCE on the 63 official validation images
from wastedet.train_det import load_detector, score_det_test_split
det, ck = load_detector(OUT / "best_det.pt", "cuda")
rep, preds, gts = score_det_test_split(det, DS.DumpsiteDetDataset(test, cfg.size), ck["score_threshold"],
                                       OUT / "test_report.json", extra={"best_epoch": ck["best_epoch"],
                                       "earlier_yolov8s_on_same_images": {"map50": 0.42, "map50_95": 0.21, "precision": 0.60, "recall": 0.36}})
print(f"Faster R-CNN: mAP50 {rep['map50']:.3f} | mAP50-95 {rep['map50_95']:.3f} | precision {rep['precision']:.3f} "
      f"| recall {rep['recall']:.3f} (score threshold {rep['score_threshold']:.2f}, from validation)")
print("earlier YOLOv8s (selected on these same images): mAP50 0.42 | mAP50-95 0.21 | precision 0.60 | recall 0.36")

In [ ]:
from wastedet import plots
FIG = OUT / "figures"; FIG.mkdir(exist_ok=True)
plots.plot_history(res["history"], FIG / "training_curves.png", keys=("val_map50",), title="Dumpsite detector - training")
order = np.argsort([-(max(s, default=0)) for _, s in preds])
for k, i in enumerate(list(order[:4]) + list(order[-2:])):
    img = (DS.DumpsiteDetDataset(test, cfg.size)[i][0].permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    plots.draw_boxes(img, gts[i], preds[i][0], preds[i][1], ck["score_threshold"], FIG / f"example_{k}.png",
                     title=test[i].image.name[:30])
from IPython.display import Image as IImage, display
for f in sorted(FIG.glob("*.png")):
    display(IImage(str(f)))
import shutil; shutil.make_archive(str(RUNS / "results_detector_v1"), "zip", OUT)